## Accelerate Inference: Neural Network Pruning

In [ ]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# # untar
# !ls
# !tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('/data/user_data/tianyuca/Courses/10605_Pruning/train_images.pkl', 'rb'))
train_labels = pickle.load(open('/data/user_data/tianyuca/Courses/10605_Pruning/train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('/data/user_data/tianyuca/Courses/10605_Pruning/val_images.pkl', 'rb'))
val_labels = pickle.load(open('/data/user_data/tianyuca/Courses/10605_Pruning/val_labels.pkl', 'rb'))

In [4]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

In [5]:
train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [7]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [27]:
model = ConvNet()
model = model.to(device)
summary(model, input_size=(3, 25, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 25, 25]             896
              ReLU-2           [-1, 32, 25, 25]               0
            Conv2d-3           [-1, 32, 23, 23]           9,248
              ReLU-4           [-1, 32, 23, 23]               0
         MaxPool2d-5           [-1, 32, 11, 11]               0
           Dropout-6           [-1, 32, 11, 11]               0
            Conv2d-7           [-1, 64, 11, 11]          18,496
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9             [-1, 64, 9, 9]          36,928
             ReLU-10             [-1, 64, 9, 9]               0
        MaxPool2d-11             [-1, 64, 4, 4]               0
          Dropout-12             [-1, 64, 4, 4]               0
          Flatten-13                 [-1, 1024]               0
           Linear-14                  [

In [28]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [29]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

In [30]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

# Main training loop
num_epochs = 50
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Training
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

Epoch 1/50


Epoch [1/50], Train Loss: 1.5189, Train Acc: 31.05%, Val Loss: 1.4058, Val Acc: 38.97%
Epoch 2/50


Epoch [2/50], Train Loss: 1.3764, Train Acc: 40.56%, Val Loss: 1.3114, Val Acc: 43.29%
Epoch 3/50


Epoch [3/50], Train Loss: 1.3218, Train Acc: 43.48%, Val Loss: 1.2667, Val Acc: 46.38%
Epoch 4/50


Epoch [4/50], Train Loss: 1.2913, Train Acc: 44.93%, Val Loss: 1.2440, Val Acc: 47.25%
Epoch 5/50


...

Epoch [46/50], Train Loss: 0.7306, Train Acc: 72.07%, Val Loss: 0.8349, Val Acc: 67.05%
Epoch 47/50


Epoch [47/50], Train Loss: 0.7234, Train Acc: 72.50%, Val Loss: 0.8190, Val Acc: 67.88%
Epoch 48/50


Epoch [48/50], Train Loss: 0.7201, Train Acc: 72.28%, Val Loss: 0.8120, Val Acc: 68.16%
Epoch 49/50


Epoch [49/50], Train Loss: 0.7097, Train Acc: 73.04%, Val Loss: 0.8126, Val Acc: 68.08%
Epoch 50/50


Epoch [50/50], Train Loss: 0.6995, Train Acc: 73.74%, Val Loss: 0.8228, Val Acc: 67.64%


# SAP Algorithm

In [31]:
import copy
from collections import OrderedDict

In [32]:
class Mask:
    """
    Tracks and applies pruning masks to model weights.
    """
    def __init__(self, state_dict: OrderedDict):
        self._mask = self._init_mask(state_dict)

    @staticmethod
    def _init_mask(state_dict: OrderedDict) -> OrderedDict:
        mask = OrderedDict()
        for name, param in state_dict.items():
            if param.dim() > 1 and 'weight' in name:
                mask[name] = param.new_ones(param.size(), dtype=torch.bool)
        return mask

    def freeze_grad(self, model: torch.nn.Module):
        """
        Zero out gradients for pruned weights in-place.
        """
        for name, param in model.named_parameters():
            if param.grad is None:
                continue
            if param.dim() > 1 and 'weight' in name:
                mask = self._mask[name].to(param.device)
                param.grad.data.mul_(mask)

    def load_state_dict(self, mask_dict: OrderedDict):
        self._mask = mask_dict

    def state_dict(self) -> OrderedDict:
        return self._mask

In [33]:
'''
SparsityIndex class to calculate the PQ Index
'''
class SparsityIndex:
    def __init__(self, p: torch.Tensor, q: torch.Tensor):
        self.p = p
        self.q = q
        self.si = {'global': []}
        self.gini = {'global': []}
    
    # Calculate the PQ Index for a given tensor
    def _calc_si(self, x: torch.Tensor, mask: torch.Tensor, dim: int, p: torch.Tensor, q: torch.Tensor):
        d = mask.to(x.device).float().sum(dim=dim)
        
        # Calculate the PQ Index
        x = x * mask.to(x.device).float()
        num = torch.linalg.norm(x, p, dim=dim).pow(p) / d
        den = torch.linalg.norm(x, q, dim=dim).pow(q) / d
        si = 1 - (num.pow(1 / p) / den.pow(1 / q))
        si[d == 0] = 0
        si[si == -float('inf')] = 0
        si[torch.logical_and(si > -1e-5, si < 0)] = 0
        return si
    
    # Calculate the Gini Index for a given tensor
    def _calc_gini(self, x: torch.Tensor, mask: torch.Tensor, dim: int):
        x = x.abs() + 1e-7
        x[~mask] = float('nan')
        x = torch.sort(x, dim=dim)[0]
        N = mask.to(x.device).float().sum(dim=dim)
        idx = torch.arange(1, x.size(dim) + 1, device=x.device)
        gini = ((2 * idx - N.view(-1, 1) - 1) * x).nansum(dim=dim) / (N * x.nansum(dim=dim))
        return gini

    def _combine_global(self, model, mask):
        params, masks = [], []
        for name, param in model.named_parameters():
            if 'weight' in name and param.dim() > 1:
                params.append(param.view(-1))
                masks.append(mask.state_dict()[name].view(-1))
        return torch.cat(params, dim=0), torch.cat(masks, dim=0)
    
    def make_sparsity_index(self, model, mask):
        param_all, mask_all = self._combine_global(model, mask)
        
        # PQ index
        si = []
        for i in range(len(self.p)):
            for j in range(len(self.q)):
                si.append(self._calc_si(param_all, mask_all, -1, self.p[i], self.q[j]))
        si = torch.tensor(si)
        si_sparsity_index = {'global': si.reshape((len(self.p), len(self.q), -1))}
        self.si['global'].append(si_sparsity_index)
        # Gini index
        gini = self._calc_gini(param_all, mask_all, -1)
        gini_sparsity_index = {'global': gini}
        self.gini['global'].append(gini_sparsity_index)

In [34]:
# SI-based bound calculation
def make_bound_si(si, d, p, q, eta_m):
    m = d * (1 + eta_m) ** (q / (p - q)) * (1 - si) ** ((q * p) / (q - p))
    return torch.ceil(m).long()

class Compression:
    '''
    Compression class to perform the actual pruning
    '''
    def __init__(self, prune_scope, prune_mode):
        super().__init__()
        self.prune_scope = prune_scope
        self.prune_mode = prune_mode
        
        self.beta = 0.9 # maximum prune ratio per iteration

    def init(self, model, mask, init_state):
        for name, param in model.named_parameters():
            data = init_state[name].to(param.device)
            if 'weight' in name and param.dim() > 1:
                mask_i = mask.state_dict()[name]
                param.data = torch.where(mask_i, data, torch.tensor(0, dtype=torch.float)).to(param.device)
            else:
                param.data = data
        return

    def compress(self, model, mask, sparsity_index):
        pivot_param = []
        pivot_mask = []
        for name, param in model.named_parameters():
            parameter_type = name.split('.')[-1]
            if 'weight' in parameter_type and param.dim() > 1:
                mask_i = mask.state_dict()[name]
                pivot_param_i = param[mask_i].abs()
                pivot_param.append(pivot_param_i.view(-1))
                pivot_mask.append(mask_i.view(-1))
        
        pivot_param = torch.cat(pivot_param, dim=0).data.abs()
        pivot_mask = torch.cat(pivot_mask, dim=0)
        
        # Calculate threshold based on pruning mode
        if self.prune_mode[0] == 'si':
            p, q, eta_m, gamma = map(float, self.prune_mode[1:])
            
            p_idx = (sparsity_index.p == p).nonzero().item()
            q_idx = (sparsity_index.q == q).nonzero().item()
            
            si_i = sparsity_index.si[self.prune_scope][-1]['global'][p_idx, q_idx]
            d = pivot_mask.float().sum().to(si_i.device)
            m = make_bound_si(si_i, d, p, q, eta_m)
            retain_ratio = m / d
            prune_ratio = torch.clamp(gamma * (1 - retain_ratio), 0, self.beta)
            num_prune = torch.floor(d * prune_ratio).long()
            pivot_value = torch.sort(pivot_param.view(-1))[0][num_prune]
        
        else:
            prune_ratio = float(self.prune_mode[1])
            pivot_value = torch.quantile(pivot_param, prune_ratio)
        
        # Apply pruning to each layer
        new_mask = OrderedDict()
        for name, param in model.named_parameters():
            parameter_type = name.split('.')[-1]
            if 'weight' in parameter_type and param.dim() > 1:
                mask_i = mask.state_dict()[name]
                pivot_mask = (param.data.abs() < pivot_value).to(mask_i.device)
                new_mask[name] = torch.where(pivot_mask, False, mask_i)
                param.data = torch.where(new_mask[name].to(param.device), 
                                            param.data,
                                            torch.tensor(0, dtype=torch.float, device=param.device)
                                            )
        
        mask.load_state_dict(new_mask)

In [35]:
# Helper function to count parameters
def count_parameters(model, mask):
    """Count total and remaining parameters in the model"""
    total_params = 0
    remaining_params = 0
    
    for name, param in model.named_parameters():
        parameter_type = name.split('.')[-1]
        if 'weight' in parameter_type and param.dim() > 1:
            if name in mask.state_dict():
                mask_i = mask.state_dict()[name]
                total_params += param.numel()
                remaining_params += mask_i.sum().item()
    
    return total_params, remaining_params


def train_one_epoch_pruning(model, train_loader, criterion, optimizer, mask, device):
    """Train the model for one epoch"""
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        
        # Zero-out gradients of pruned weights
        mask.freeze_grad(model)

        optimizer.step()
    return


'''
Main function to apply SAP to a model
'''
def apply_sap(model, train_loader, val_loader, p=0.5, q=1.0, eta_m=0, gamma=1.0,
              prune_scope='global', prune_iterations=50, device='cuda'):
    # Initialize p and q tensors for the sparsity index
    p_values = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    q_values = torch.tensor([1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0])
    
    # Initialize sparsity index calculator
    sparsity_index = SparsityIndex(p_values, q_values)

    # Initialize model and mask
    model = model.to(device)
    mask = Mask(model.state_dict())
    
    # Initialize pruning configuration
    prune_mode = ['si', p, q, eta_m, gamma]
    compression = Compression(prune_scope, prune_mode)
    
    # History tracking
    pruning_history = {
        'accuracy': [],
        'remaining_weights': [],
        'pq_index': []
    }
    
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)
    
    # Training loop
    for iteration in range(prune_iterations + 1):
        print(f"Iteration {iteration}/{prune_iterations}")
        
        accuracy = validate(model, val_loader, criterion, device)[1]
        sparsity_index.make_sparsity_index(model, mask)
        total_params, remaining_params = count_parameters(model, mask)
        
        # Get current PQ index
        p_idx = (sparsity_index.p == p).nonzero()[0].item()
        q_idx = (sparsity_index.q == q).nonzero()[0].item()
        current_pq_index = sparsity_index.si[prune_scope][-1]['global'][p_idx, q_idx].item()
        
        # Record history
        pruning_history['accuracy'].append(accuracy)
        pruning_history['remaining_weights'].append(remaining_params / total_params)
        pruning_history['pq_index'].append(current_pq_index)
        
        print(f"Accuracy: {accuracy:.2f}%, Remaining weights: {remaining_params/total_params:.2f}, PQ Index: {current_pq_index:.4f}")
        
        # Exit if last iteration
        if iteration == prune_iterations:
            break
        
        # Prune the model
        compression.compress(model, mask, sparsity_index)
        
        # Train for one epoch
        train_one_epoch_pruning(model, train_loader, criterion, optimizer, mask, device)
        scheduler.step()
    
    return model, pruning_history, mask

In [36]:
# model = ConvNet()
# use the pretrained model instead

# Load the pretrained model
model_to_prune = copy.deepcopy(model)
model_to_prune.load_state_dict(model.state_dict())
# Apply SAP
pruned_model, history, mask = apply_sap(
    model_to_prune,
    train_loader,
    val_loader,
    p=0.5,          # p parameter for PQ Index
    q=1.0,          # q parameter for PQ Index
    eta_m=0,        # Regularization parameter
    gamma=1.0,      # Pruning aggressiveness
    prune_scope='global',  # 'global'
    prune_iterations=50,  # Number of pruning iterations
    device='cuda'   # Device to use for computation
)

# Print final statistics
print("Final pruning statistics:")
print(f"Accuracy: {history['accuracy'][-1]:.2f}%")
print(f"Compression ratio: {1.0 / history['remaining_weights'][-1]:.2f}")
print(f"Final PQ Index: {history['pq_index'][-1]:.4f}")

Iteration 0/50


Accuracy: 67.64%, Remaining weights: 1.00, PQ Index: 0.2250


Iteration 1/50


Accuracy: 54.38%, Remaining weights: 0.78, PQ Index: 0.1235


Iteration 2/50


...

Accuracy: 70.69%, Remaining weights: 0.03, PQ Index: 0.0472


Iteration 49/50


Accuracy: 69.74%, Remaining weights: 0.03, PQ Index: 0.0468


Iteration 50/50


Accuracy: 69.23%, Remaining weights: 0.03, PQ Index: 0.0461
Final pruning statistics:
Accuracy: 69.23%
Compression ratio: 37.14
Final PQ Index: 0.0461


In [37]:
def cal_score(acc, remaining_params, total_params):
    return (acc + (total_params - remaining_params) / total_params) / 2

In [38]:
total_params, remaining_params = count_parameters(pruned_model, mask)
score = cal_score(history['accuracy'][-1], remaining_params, total_params)
print(f"Total parameters: {total_params}")
print(f"Remaining parameters: {remaining_params}")
print(f"Score: {score:.4f}")

Total parameters: 592224
Remaining parameters: 15945
Score: 35.1004


In [43]:
torch.save(model.state_dict(), f'my_model_weights_acc{history["accuracy"][-1]:.2f}_score{score:.2f}.pt', _use_new_zipfile_serialization=False)

In [44]:
del model
del pruned_model
torch.cuda.empty_cache()